# FaceFill

Predict upper-face ARKit morphs (brows / eyes / blinks) from mouth / lower-face when a VR headset hides the upper face.

```
MEAD (person-disjoint) → NPZ → Dataset windows
  → AvatarFaceModel (encoders → GRU → heads)
  → Stage 1 teacher-forced → Stage 2 scheduled sampling
  → artifacts/model/best.pt (+ ONNX)
```

This checkout trains on **3 MEAD actors** (M033 / W015 / W009). See §2 for the full data mix.


## 0. Setup

In [ ]:
from pathlib import Path
import os, sys, json

ROOT = Path.cwd()
if not (ROOT / "src" / "avatarface").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

SEQ_ROOT = ROOT / "data/processed/sequences"
SPLITS = ROOT / "data/splits/splits.json"
RAW_MEAD = ROOT / "data/raw/mead"
MP_MODEL = ROOT / "assets/face_landmarker.task"
OUT = ROOT / "artifacts/model"

RUN_EXTRACT = False
RUN_SPLITS = True
RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_EVAL = True

STAGE1_EPOCHS = 30
STAGE2_EPOCHS = 20
BATCH = 16
LR = 3e-4
SEQ_LEN = 128
BURN_IN = 32
ROLLOUT_HORIZON = 60
ROLLOUT_WEIGHT = 0.5

import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np

from avatarface.data.dataset import MorphCompletionDataset
from avatarface.data.splits import build_splits
from avatarface.models.avatarface_model import AvatarFaceModel
from avatarface.training.losses import CompletionLoss
from avatarface.training.scheduled_sampling import teacher_forcing_prob
from avatarface.training.evaluate import evaluate_checkpoint, latency_benchmark, load_model
from avatarface.runtime.export import export_onnx
from avatarface.morphs.arkit_schema import ARKIT_52, ARKIT_INDEX, N_FULL
from avatarface.morphs.channel_groups import (
    LOWER_FACE_CHANNELS, UPPER_FACE_CHANNELS, LOWER_IDX, UPPER_IDX,
    N_CURRENT, N_HISTORY, N_UPPER, N_LOWER,
)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

OUT.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)
print("device", DEVICE)

## 1. Morph

MediaPipe and ARKit expose **52** named blendshapes. FaceFill partitions them as

| Group | Count | Role |
|-------|------:|------|
| Lower | 30 | Observed under an HMD (jaw, mouth, cheeks, nose sneer) |
| Upper | 11 | **Predicted** (blinks, squint, wide, brows) |
| Gaze | 8 | External — not predicted from mouth |
| Extra | 3 | cheekSquint L/R, tongueOut |

Order is locked in `arkit_schema.py`. 

In [ ]:
print(f"ARKIT_52 = {N_FULL} channels")
print("lower (30):", LOWER_FACE_CHANNELS[:5], "…")
print("upper (11):", UPPER_FACE_CHANNELS)
print()
print("index examples:")
for name in ("jawOpen", "mouthSmileLeft", "eyeBlinkLeft", "browInnerUp"):
    print(f"  {name:16s} → full[{ARKIT_INDEX[name]}]")

## 2. Data: video → NPZ → person splits

### Source
**MEAD** frontal videos only (`camera=front`). Each clip → one NPZ:

| Field | Shape / type | Meaning |
|-------|--------------|---------|
| `full_morphs` | `[T, 52]` float | MediaPipe ARKit blendshapes |
| `lower_morphs` | `[T, 30]` | lower slice (also recomputed in dataset) |
| `head_features` | `[T, 12]` | head pose features |
| `tracking_confidence` | `[T, 3]` | tracker confidence |
| `meta_json` | string | `person_id`, emotion, intensity, fps, … |

### Current data mix (this checkout)
Person disjoint, 1 actor per split (full MEAD blocked by Drive quota):

| Split | Actor | Clips | ~frames | Notes |
|-------|-------|------:|--------:|-------|
| train | **M033** | 659 | 91k | learn mapping |
| val | **W015** | 664 | 82k | pick checkpoints |
| test | **W009** | 656 | 90k | final report |

Emotions (≈balanced across splits), angry, contempt, disgusted, fear, happy, sad, surprised (~80–90 clips each); neutral thinner (~40). Clip length ≈ 33–301 frames (mean ~130 @ 30 fps). MEAD intensity levels stay in meta but are not stratified.

### Split rules
- By person_id, never random frames across people
- Same actor never in train and test, no cross-identity generalization
- 3 actors = MVP / smoke fit; more identities → better brows/blinks

### Scaling up
Add MEAD actor tars under `data/raw/mead/`, set `RUN_EXTRACT=True`, rebuild splits. Target when storage allows: many train IDs, held-out val/test people (~80/10/10 by person).


In [ ]:
if RUN_EXTRACT:
    assert RAW_MEAD.is_dir(), f"missing {RAW_MEAD}"
    if not MP_MODEL.exists():
        import urllib.request
        MP_MODEL.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(
            "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
            MP_MODEL,
        )
    from avatarface.data.extract_video import main as extract_main
    import sys as _sys
    _sys.argv = [
        "extract", "--dataset", "mead", "--camera", "front",
        "--model", str(MP_MODEL), "--out", str(SEQ_ROOT),
    ]
    extract_main()

npz_paths = sorted(SEQ_ROOT.rglob("*.npz"))
assert npz_paths, f"no NPZs under {SEQ_ROOT}"
print(f"NPZs: {len(npz_paths)}")

example = np.load(npz_paths[0], allow_pickle=True)
full = example["full_morphs"]
meta = json.loads(str(example["meta_json"]))
print("example file:", npz_paths[0].name)
print("meta:", {k: meta.get(k) for k in ("person_id", "emotion", "camera", "fps")})
print("full_morphs shape:", full.shape)
print("jawOpen mean:", float(full[:, ARKIT_INDEX["jawOpen"]].mean()))
print("eyeBlinkLeft mean:", float(full[:, ARKIT_INDEX["eyeBlinkLeft"]].mean()))

if RUN_SPLITS or not SPLITS.exists():
    splits = build_splits(SEQ_ROOT, seed=0)
    SPLITS.parent.mkdir(parents=True, exist_ok=True)
    SPLITS.write_text(json.dumps(splits, indent=2))
else:
    splits = json.loads(SPLITS.read_text())

print("people:", splits.get("people"))
print({k: len(splits[k]) for k in ("train", "val", "test")})

# live data-mix summary
from collections import Counter
print("\n--- data mix ---")
for split_name in ("train", "val", "test"):
    rows = splits[split_name]
    people = Counter(r["person_id"] for r in rows)
    emotions = Counter()
    n_frames = 0
    for r in rows:
        p = SEQ_ROOT / r["path"]
        if not p.exists():
            continue
        z = np.load(p, allow_pickle=True)
        m = json.loads(str(z["meta_json"]))
        emotions[m.get("emotion", "?")] += 1
        n_frames += int(z["full_morphs"].shape[0])
    print(f"{split_name:5s}  people={dict(people)}  clips={len(rows)}  frames={n_frames}")
    print(f"       emotions={dict(sorted(emotions.items()))}")


## 3. Dataset windows

`MorphCompletionDataset` cuts each clip into length-`SEQ_LEN` windows (default **128**), stride `SEQ_LEN // 2`.

| Tensor | Shape | Meaning |
|--------|-------|---------|
| `current` | `[T, 80]` | lower30 + lower_vel30 + head12 + conf3 + blink_state5 |
| `history` | `[T, 176]` | prev full52 + vel52 + HMD obs_mask52 + head12 + conf3 + blink5 |
| `upper` | `[T, 11]` | GT upper targets |
| `upper_prev` | `[T, 11]` | previous upper (teacher signal) |
| `blink_targets` | `[T, 5]` | onset L/R/bi, duration, amplitude |
| `supervise` | `[T]` | 0 for first `BURN_IN` (32) frames, 1 after |

**HMD mask:** lower (+ `tongueOut`) observed; upper / gaze / cheekSquint unobserved → model must fill them.

**Train augment:** light noise / dropout on lower (`augment_lower`).

**Window math:** a 138-frame clip → starts at 0, 64 → up to 2 windows of 128 (clips shorter than 128 are skipped).


In [ ]:
def load_split(name):
    return json.loads(SPLITS.read_text())[name]

train_rows, val_rows = load_split("train"), load_split("val")
train_ds = MorphCompletionDataset(SEQ_ROOT, train_rows, SEQ_LEN, BURN_IN, augment=True)
val_ds = MorphCompletionDataset(
    SEQ_ROOT, val_rows or train_rows, SEQ_LEN, BURN_IN, augment=False
)
assert len(train_ds) > 0

sample = train_ds[0]
print(f"windows train={len(train_ds)} val={len(val_ds)}")
print("one window:")
for k, v in sample.items():
    print(f"  {k:14s} {tuple(v.shape)}")

assert sample["current"].shape[-1] == N_CURRENT == 80
assert sample["history"].shape[-1] == N_HISTORY == 176
assert sample["upper"].shape[-1] == N_UPPER == 11

train_loader = DataLoader(train_ds, batch_size=min(BATCH, len(train_ds)), shuffle=True)
val_loader = DataLoader(val_ds, batch_size=min(BATCH, max(1, len(val_ds))), shuffle=False)
batch = next(iter(train_loader))
print("batched current:", tuple(batch["current"].shape), "  # [B, T, 80]")

## 4. Model

```
current[80]  → MLP → 256 ─┐
history[176] → MLP → 256 ─┴→ fuse → 384 → GRU×3 (hidden 384)
                                              │
                         ┌────────────────────┴────────────────────┐
                         │                    │                    │
                      AbsHead              ResHead              ┌──┴──┐
                      u_abs[11]            u_delta[11]       UncHead  BlinkHead
                         │                    │              log_var  onset/dur/amp
                         └─────────┬──────────┘                 │         │
                                   ▼                         (loss)   (blink ctrl)
                    u_hat = clamp( α·(u_prev + u_delta) + (1−α)·u_abs )
```

- `forward(...)` — whole sequence (Stage 1 teacher-forced)
- `step(...)` — one frame + GRU hidden (Stage 2, demo, ONNX)

`u_prev[11]` is an extra input used only in the combine formula (α=0.8). Blink/Unc do **not** feed `u_hat`.


In [ ]:
model = AvatarFaceModel().to(DEVICE)
nparams = sum(p.numel() for p in model.parameters())
print(f"params={nparams:,}")

B, T = 2, 16
cur = torch.randn(B, T, N_CURRENT, device=DEVICE)
hist = torch.randn(B, T, N_HISTORY, device=DEVICE)
u_prev = torch.rand(B, T, N_UPPER, device=DEVICE)

with torch.no_grad():
    out = model(cur, hist, u_prev)

print("forward outputs:")
for k, v in out.items():
    if torch.is_tensor(v):
        print(f"  {k:20s} {tuple(v.shape)}")

with torch.no_grad():
    step_out = model.step(cur[:, 0], hist[:, 0], u_prev[:, 0], hidden=None)
print("step u_hat:", tuple(step_out["u_hat"].shape), "  # [B, 11]")
print("step hidden:", tuple(step_out["hidden"].shape), "  # [layers, B, 384]")

## 5. Loss

`CompletionLoss` weights:

| Term | Weight | What it does |
|------|-------:|--------------|
| absolute | 1.0 | Smooth-L1 `u_abs` vs GT upper |
| residual | 1.0 | Smooth-L1 `u_delta` vs `upper − upper_prev` |
| velocity | 0.5 | Match frame-to-frame Δ upper |
| acceleration | 0.1 | Second-order smoothness |
| blink | 1.0 | Onset BCE (L/R/bilateral) + dur/amp |
| uncertainty | 0.25 | NLL on `log_var` around `u_abs` |
| range | 0.05 | Keep `u_hat` in `[0, 1]` |

Only `supervise=1` frames (after burn-in) count. Stage 2 adds `ROLLOUT_WEIGHT ×` AR rollout loss over `ROLLOUT_HORIZON` (60) frames.


In [ ]:
criterion = CompletionLoss()
demo_batch = {k: v[:2, :16].to(DEVICE) for k, v in batch.items()}
with torch.no_grad():
    demo_out = model(demo_batch["current"], demo_batch["history"], demo_batch["upper_prev"])
    loss, parts = criterion(demo_out, demo_batch)

print(f"loss = {float(loss):.4f}")
print("parts:", {k: round(float(v), 4) for k, v in parts.items()})

## 6. Three forward modes

| Mode | `u_prev` source | Used when |
|------|-----------------|-----------|
| Teacher-forced | Ground-truth upper | Stage 1 train + val |
| Scheduled sampling | GT with prob `p_tf`, else model prediction | Stage 2 train |
| Autoregressive rollout | Always model prediction | Stage 2 val, deploy |

Stage 2 anneals `p_tf` so the net learns to survive its own errors — matching headset runtime.

In [ ]:
def forward_teacher(model, batch):
    return model(batch["current"], batch["history"], batch["upper_prev"])


def forward_scheduled(model, batch, p_tf: float):
    current, history, upper = batch["current"], batch["history"], batch["upper"]
    bsz, T, _ = current.shape
    hidden, u_prev = None, batch["upper_prev"][:, 0]
    buckets = {k: [] for k in ("u_hat", "u_abs", "u_delta", "log_var",
                               "blink_onset_logits", "blink_duration", "blink_amplitude")}
    for t in range(T):
        out = model.step(current[:, t], history[:, t], u_prev, hidden=hidden)
        hidden = out["hidden"]
        for k in buckets:
            buckets[k].append(out[k])
        use_tf = (torch.rand(bsz, device=current.device) < p_tf).unsqueeze(-1)
        u_prev = torch.where(use_tf, upper[:, t], out["u_hat"].detach())
    return {k: torch.stack(v, dim=1) for k, v in buckets.items()}


def forward_rollout(model, batch, horizon: int):
    current, history = batch["current"], batch["history"]
    H = min(horizon, current.shape[1])
    hidden, u_prev = None, batch["upper_prev"][:, 0]
    buckets = {k: [] for k in ("u_hat", "u_abs", "u_delta", "log_var",
                               "blink_onset_logits", "blink_duration", "blink_amplitude")}
    for t in range(H):
        out = model.step(current[:, t], history[:, t], u_prev, hidden=hidden)
        hidden = out["hidden"]
        for k in buckets:
            buckets[k].append(out[k])
        u_prev = out["u_hat"]
    return {k: torch.stack(v, dim=1) for k, v in buckets.items()}, H


tiny = {k: v[:1, :8].to(DEVICE) for k, v in batch.items()}
with torch.no_grad():
    tf = forward_teacher(model, tiny)
    ss = forward_scheduled(model, tiny, p_tf=0.5)
    ar, H = forward_rollout(model, tiny, horizon=8)

print("teacher  u_hat", tuple(tf["u_hat"].shape))
print("scheduled u_hat", tuple(ss["u_hat"].shape), "  p_tf=0.5")
print("rollout   u_hat", tuple(ar["u_hat"].shape), f"  H={H}")
print("p_tf schedule examples:", [teacher_forcing_prob(e) for e in (1, 5, 10, 20)])

## 7. Train loop

One function covers both stages. Stage 2 adds a short AR rollout loss so long closed-loop runs stay stable.

In [ ]:
def train_stage(
    stage: int,
    epochs: int,
    *,
    init_ckpt=None,
    rollout_horizon: int = 0,
    rollout_weight: float = 0.0,
):
    model = AvatarFaceModel().to(DEVICE)
    if init_ckpt and Path(init_ckpt).exists():
        blob = torch.load(init_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(blob["model"])
        print(f"warm-start {init_ckpt}")

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, betas=(0.9, 0.95))
    criterion = CompletionLoss()
    nparams = sum(p.numel() for p in model.parameters())
    print(f"stage={stage} params={nparams:,} epochs={epochs} device={DEVICE}")

    best, history = float("inf"), []
    for epoch in range(1, epochs + 1):
        p_tf = 1.0 if stage == 1 else teacher_forcing_prob(epoch)
        model.train()
        tr, n = 0.0, 0
        for batch in tqdm(train_loader, desc=f"s{stage} ep{epoch} tf={p_tf:.2f}", leave=False):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            if stage == 1:
                out = forward_teacher(model, batch)
                loss, _ = criterion(out, batch)
            else:
                out = forward_scheduled(model, batch, p_tf)
                loss, _ = criterion(out, batch)
                if rollout_horizon > 0 and rollout_weight > 0:
                    roll_out, H = forward_rollout(model, batch, rollout_horizon)
                    roll_batch = {
                        "upper": batch["upper"][:, :H],
                        "upper_prev": batch["upper_prev"][:, :H],
                        "blink_targets": batch["blink_targets"][:, :H],
                        "supervise": batch["supervise"][:, :H],
                    }
                    roll_loss, _ = criterion(roll_out, roll_batch)
                    loss = loss + rollout_weight * roll_loss

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += float(loss.detach()) * batch["current"].shape[0]
            n += batch["current"].shape[0]
        tr /= max(n, 1)

        model.eval()
        va, nv = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                if stage == 1:
                    out = forward_teacher(model, batch)
                else:
                    out, _ = forward_rollout(model, batch, batch["current"].shape[1])
                loss, _ = criterion(out, batch)
                va += float(loss) * batch["current"].shape[0]
                nv += batch["current"].shape[0]
        va /= max(nv, 1)
        history.append({"epoch": epoch, "train": tr, "val": va, "p_tf": p_tf, "stage": stage})
        print(f"epoch {epoch:03d}  train {tr:.4f}  val {va:.4f}  p_tf {p_tf:.2f}")

        ckpt = {
            "model": model.state_dict(), "epoch": epoch, "val": va,
            "nparams": nparams, "stage": stage, "p_tf": p_tf,
        }
        torch.save(ckpt, OUT / "last.pt")
        if va < best:
            best = va
            torch.save(ckpt, OUT / "best.pt")
            print(f"  saved best.pt ({best:.4f})")

    (OUT / f"history_stage{stage}.json").write_text(json.dumps(history, indent=2))
    print(f"stage {stage} done → {OUT / 'best.pt'}  best_val={best:.4f}")
    return OUT / "best.pt"

## 8. Stage 1 → Stage 2

| | Stage 1 | Stage 2 |
|--|---------|---------|
| Init | random | warm-start `best.pt` |
| `u_prev` | always GT | GT with prob `p_tf`, else prediction |
| Extra loss | — | AR rollout × 0.5 over H=60 |
| Val | teacher-forced | full autoregressive |
| Epochs (default) | 30 | 20 |

**`p_tf` schedule:** epoch 1→0.95 · 3→0.80 · 5→0.60 · 8→0.40 · ≥20→0.10

Optimizer: AdamW `lr=3e-4`, `weight_decay=0.01`, grad clip 1.0, batch 16.


In [ ]:
if RUN_STAGE1:
    train_stage(1, STAGE1_EPOCHS)
else:
    assert (OUT / "best.pt").exists()
    print("skip stage1 →", OUT / "best.pt")

if RUN_STAGE2:
    train_stage(
        2,
        STAGE2_EPOCHS,
        init_ckpt=OUT / "best.pt",
        rollout_horizon=ROLLOUT_HORIZON,
        rollout_weight=ROLLOUT_WEIGHT,
    )
else:
    print("skip stage2 →", OUT / "best.pt")

## 9. Eval + ONNX

Test metrics are autoregressive. ONNX exports a single `step` with hidden state I/O for the browser / WebXR companion.

In [ ]:
ckpt = OUT / "best.pt"
assert ckpt.exists(), ckpt

if RUN_EVAL:
    metrics = evaluate_checkpoint(
        ckpt, SEQ_ROOT, SPLITS, split="test", batch=8,
        device=str(DEVICE), autoregressive=True,
    )
    model = load_model(ckpt, DEVICE)
    metrics["latency"] = latency_benchmark(model, DEVICE)
    (OUT / "eval.json").write_text(json.dumps(metrics, indent=2))
    print(
        f"MAE {metrics['mae_mean']:.4f}  Pearson {metrics['pearson_mean']:.4f}  "
        f"blink F1 {metrics['blink_onset']['f1']:.4f}  "
        f"step_ms {metrics['latency']['step_ms']:.3f}"
    )
    print("per-group MAE:", {k: round(v, 4) for k, v in metrics["mae_groups"].items()})

    onnx_path = export_onnx(ckpt, OUT / "model.onnx")
    print("onnx", onnx_path)

print("done →", ckpt)
print("local demo:  ./mvp demo")
print("VR:          WebXREmotion companion loads assets/facefill/model.onnx")